# Tool Schemas, Validation, and Agent Skills

Building on how tools work, this notebook focuses on **how to design tools well**, schemas that constrain inputs, validation that catches errors before they propagate, and the concept of grouping related tools into reusable **agent skills**.

Use case: Competitor intelligence pipeline

We're preparing for a competitive deal against NexaCRM at Hartwell Financial Group (a $500K enterprise CRM contract). The agent must gather: pricing tiers, capability gaps, recent product updates, and customer sentiment, then produce a battle-card summary.

**Why this use case?** Competitive intelligence involves structured, repeatable lookups across predictable categories (pricing, features, news, reviews). This makes it ideal for demonstrating constrained schemas with enum-like inputs, validation logic, and skill grouping.

We will
- Design schemas with enum constraints, precise descriptions, and appropriate required fields
- Implement input validation that returns structured error messages (not Python exceptions)
- Understand how bad schema design leads to agent errors
- Group related tools into an **agent skill**, a reusable, self-contained capability module

## Setup

In [16]:
import os
import google.genai as genai

client = genai.Client() # api key in .env (loaded)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


---
## The Task


In [1]:
TASK = (
    "We're going up against NexaCRM for a $500K enterprise CRM deal at Hartwell Financial Group. "
    "Build a competitive battle card. I need: "
    "(1) their three pricing tiers with key feature differences, "
    "(2) top capability gaps in analytics and AI, "
    "(3) any product updates in the last 6 months, "
    "(4) a summary of negative customer sentiment from enterprise buyers, "
    "(5) two or three talking points we can use in the deal."
)
print("Task:", TASK)

Task: We're going up against NexaCRM for a $500K enterprise CRM deal at Hartwell Financial Group. Build a competitive battle card. I need: (1) their three pricing tiers with key feature differences, (2) top capability gaps in analytics and AI, (3) any product updates in the last 6 months, (4) a summary of negative customer sentiment from enterprise buyers, (5) two or three talking points we can use in the deal.


---
## Part 1: Schema Design

### Why schemas matter

The schema is the contract between the LLM and the tool. A poorly designed schema causes:
- The LLM passing invalid values (e.g. `tier="premium"` when only `"enterprise"` exists)
- Ambiguous calls that return wrong data
- The agent wasting steps recovering from tool errors

### Design principles demonstrated here

1. **Enum constraints in descriptions**: list valid values explicitly so the LLM can't guess wrong
2. **Precise, non-overlapping descriptions**: each tool should have exactly one right use case
3. **Fail with information, not exceptions**: return a structured error string, not a stack trace


In [2]:
# Tool implementations with input validation

VALID_TIERS      = ("starter", "professional", "enterprise")
VALID_CATEGORIES = ("core", "integration", "analytics", "ai")
VALID_SENTIMENTS = ("positive", "negative", "mixed")


def get_product_pricing(vendor: str, tier: str) -> str:
    # Returns pricing and included features for a specific tier.
    # Validates tier against allowed values before lookup.
    tier = tier.lower().strip()
    if tier not in VALID_TIERS:
        return (
            f"VALIDATION_ERROR: invalid tier '{tier}'. "
            f"Valid values are: {VALID_TIERS}. "
            "Call this function again with a valid tier."
        )
    if vendor.lower() == "nexacrm":
        data = {
            "starter": (
                "NexaCRM Starter: $49/user/month (max 10 users) | "
                "Includes: contact management, pipeline view, email integration, basic dashboards | "
                "Excludes: API access, custom reports, SSO, dedicated support"
            ),
            "professional": (
                "NexaCRM Professional: $89/user/month | "
                "Includes: all Starter + API access, advanced pipeline analytics, "
                "Salesforce/HubSpot connectors, Slack/Jira integration | "
                "Excludes: SSO, SLA, dedicated CSM, enterprise security controls"
            ),
            "enterprise": (
                "NexaCRM Enterprise: custom pricing (avg $120-150/user/month) | "
                "Includes: all Professional + SSO (SAML/OIDC), SLA 99.9%, dedicated CSM, "
                "audit logs, custom data retention, multi-region support | "
                "Note: AI features (deal scoring) are add-on at $15/user/month"
            ),
        }
        return data[tier]
    return f"No pricing data for vendor '{vendor}'"


def get_feature_analysis(vendor: str, category: str) -> str:
    # Returns strengths and weaknesses for a feature category.
    category = category.lower().strip()
    if category not in VALID_CATEGORIES:
        return (
            f"VALIDATION_ERROR: invalid category '{category}'. "
            f"Valid values are: {VALID_CATEGORIES}."
        )
    if vendor.lower() == "nexacrm":
        data = {
            "core": (
                "Core CRM: Strengths: clean modern UI, fast onboarding (avg 2 days), "
                "strong mobile app (iOS + Android). "
                "Weaknesses: no native CPQ/quoting module, limited territory management, "
                "no workflow automation below Enterprise tier."
            ),
            "integration": (
                "Integrations: 47 native connectors including Salesforce, HubSpot, Slack, "
                "Jira, Zendesk, Microsoft 365. "
                "Weaknesses: poor SAP/ERP support (no certified connector), "
                "Salesforce bi-directional sync has known data duplication bug (open since Q1 2024)."
            ),
            "analytics": (
                "Analytics: pre-built dashboards for pipeline, activity, forecast. "
                "Custom report builder available (Professional+). "
                "Weaknesses: no predictive analytics, no cohort analysis, "
                "data export limited to CSV/Excel (no direct BI connector), "
                "dashboard refresh delay up to 4 hours on large datasets."
            ),
            "ai": (
                "AI: deal scoring (beta, accuracy ~68% in backtests), "
                "email reply suggestions (available Professional+). "
                "Weaknesses: no conversational AI, no churn prediction, "
                "AI features are add-on (not included in Enterprise base price). "
                "Roadmap: AI forecasting promised Q3 2025 -- repeatedly delayed."
            ),
        }
        return data[category]
    return f"No feature data for '{vendor}'"


def get_recent_news(vendor: str, months_back: int) -> str:
    # Returns notable product/company news within the specified lookback window.
    if not (1 <= months_back <= 24):
        return (
            f"VALIDATION_ERROR: months_back must be between 1 and 24. Got {months_back}."
        )
    if vendor.lower() == "nexacrm":
        return (
            f"NexaCRM news (last {months_back} months): "
            "Series C $45M closed April 2024 (led by Accel). "
            "NexaAI assistant launched June 2024 (limited beta, invite-only). "
            "Lost Barclays $2M deal to Salesforce (reported July 2024 via LinkedIn). "
            "Two senior Product Managers left for a competitor (August 2024). "
            "New VP of Enterprise Sales hired (former Salesforce, September 2024). "
            "UK expansion office opened Manchester (October 2024)."
        )
    return f"No news for '{vendor}'"


def get_customer_sentiment(vendor: str, segment: str, sentiment: str) -> str:
    # Returns aggregated customer review themes for a market segment and sentiment.
    sentiment = sentiment.lower().strip()
    if sentiment not in VALID_SENTIMENTS:
        return (
            f"VALIDATION_ERROR: invalid sentiment '{sentiment}'. "
            f"Valid values are: {VALID_SENTIMENTS}."
        )
    if vendor.lower() == "nexacrm":
        if sentiment == "negative":
            return (
                f"NexaCRM negative reviews ({segment}): "
                "Enterprise buyers frequently cite: 'reporting too basic for our scale', "
                "'no native CPQ is a dealbreaker', 'SSO on Azure AD is unreliable', "
                "'AI features feel like marketing -- not production-ready'. "
                "12% of reviews mention data export limitations. "
                "G2 rating trend: declining from 4.4 to 4.1 over last 18 months in the 200+ user segment."
            )
        if sentiment == "positive":
            return (
                f"NexaCRM positive reviews ({segment}): "
                "'Fastest CRM onboarding we have done', 'support team genuinely responsive', "
                "'mobile app is best in class'. G2 overall 4.3/5 (620 reviews)."
            )
        return (
            f"NexaCRM mixed reviews ({segment}): "
            "Mid-market buyers (50-200 seats) most mixed. Happy with core UX, "
            "frustrated by analytics gaps and high Enterprise uplift for basic features."
        )
    return f"No review data for '{vendor}'"


TOOLS = {
    "get_product_pricing":   get_product_pricing,
    "get_feature_analysis":  get_feature_analysis,
    "get_recent_news":       get_recent_news,
    "get_customer_sentiment": get_customer_sentiment,
}

# Demo the validation guard
print("Valid call:")
print(get_product_pricing("NexaCRM", "enterprise"))
print()
print("Invalid tier -- agent receives error, not exception:")
print(get_product_pricing("NexaCRM", "premium"))


Valid call:
NexaCRM Enterprise: custom pricing (avg $120-150/user/month) | Includes: all Professional + SSO (SAML/OIDC), SLA 99.9%, dedicated CSM, audit logs, custom data retention, multi-region support | Note: AI features (deal scoring) are add-on at $15/user/month

Invalid tier -- agent receives error, not exception:
VALIDATION_ERROR: invalid tier 'premium'. Valid values are: ('starter', 'professional', 'enterprise'). Call this function again with a valid tier.


---
## Part 2: Annotated Tool Schemas

The schemas below encode the validation constraints so the LLM knows valid inputs
*before* it calls the function. When constraints are in the description, the LLM
self-corrects at planning time rather than discovering errors at execution time.


In [2]:
from google.genai import types

TOOL_DEFINITIONS = {
    "function_declarations": [

        {
            "name": "get_product_pricing",
            "description": (
                "Returns pricing and included features for a specific product tier. "
                "tier must be exactly one of: 'starter', 'professional', 'enterprise'."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "vendor": {
                        "type": "string",
                        "description": "Vendor name, e.g. NexaCRM."
                    },
                    "tier": {
                        "type": "string",
                        "enum": ["starter", "professional", "enterprise"],
                        "description": "Pricing tier."
                    }
                },
                "required": ["vendor", "tier"]
            }
        },

        {
            "name": "get_feature_analysis",
            "description": (
                "Returns strengths and weaknesses for a feature category. "
                "category must be exactly one of: 'core', 'integration', 'analytics', 'ai'."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "vendor": {
                        "type": "string",
                        "description": "Vendor name."
                    },
                    "category": {
                        "type": "string",
                        "enum": ["core", "integration", "analytics", "ai"],
                        "description": "Feature category."
                    }
                },
                "required": ["vendor", "category"]
            }
        },

        {
            "name": "get_recent_news",
            "description": (
                "Returns notable product and company news within the last N months. "
                "months_back must be an integer between 1 and 24."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "vendor": {
                        "type": "string",
                        "description": "Vendor name."
                    },
                    "months_back": {
                        "type": "integer",
                        "minimum": 1,
                        "maximum": 24,
                        "description": "Lookback window in months (1-24). Use 6 for recent changes."
                    }
                },
                "required": ["vendor", "months_back"]
            }
        },

        {
            "name": "get_customer_sentiment",
            "description": (
                "Returns aggregated customer review themes for a market segment and sentiment type. "
                "sentiment must be exactly one of: 'positive', 'negative', 'mixed'."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "vendor": {
                        "type": "string",
                        "description": "Vendor name."
                    },
                    "segment": {
                        "type": "string",
                        "description": "Market segment, e.g. 'enterprise' or 'mid-market'."
                    },
                    "sentiment": {
                        "type": "string",
                        "enum": ["positive", "negative", "mixed"],
                        "description": "Sentiment filter."
                    }
                },
                "required": ["vendor", "segment", "sentiment"]
            }
        },

    ]
}
print("Schema defined with", len(TOOL_DEFINITIONS['function_declarations']), "tools.")

Schema defined with 4 tools.


---
## Part 3: Agent Skills

An **agent skill** is a reusable, self-contained capability module: a named set of tools,
a system prompt, and configuration that together accomplish a specific class of task.

Instead of creating a new agent from scratch every time you need competitor intelligence,
you package the tools, prompt, and model configuration as a `CompetitorIntelligenceSkill`
class that can be instantiated with any vendor name.

**Benefits of the skill pattern:**
- **Reusability**: the same skill runs against any vendor with one argument change
- **Testability**: skills can be unit-tested in isolation
- **Composability**: a higher-level agent can delegate to multiple specialised skills
- **Ownership**: each skill has a clear scope and can be maintained independently


In [52]:
class CompetitorIntelligenceSkill:
    """
    Agent skill for competitive intelligence.
    Given a competitor name and a deal context, produces a battle card.
    """

    def model(self, msg, tool_defs = TOOL_DEFINITIONS):
        return client.models.generate_content(model='gemini-2.5-flash-lite',
                                            contents=msg,
                                            config = types.GenerateContentConfig(
                                            system_instruction=(
                                                    "You are a competitive intelligence analyst. "
                                                    "Use the tools to research the competitor systematically: pricing tiers first, "
                                                    "then capabilities by category (analytics and ai are highest priority), "
                                                    "then recent news, then negative enterprise sentiment. "
                                                    "Produce a structured battle card with clear talking points."
                                                ),
                                            tools=[tool_defs]))


    def __init__(self, tool_definitions: dict = TOOLS, max_steps: int = 12):
        self.tool_definitions = tool_definitions
        self.max_steps       = max_steps

    def run(self, vendor: str, deal_context: str) -> str:
        goal = (
            f"Build a competitive battle card for vendor '{vendor}'. "
            f"Deal context: {deal_context}"
        )
        messages = [{"role": "user", "parts": [{"text": goal}]}]
        step = 0

        while step < self.max_steps:
            step += 1
            print(f"  [Skill step {step}] Calling LLM...")
            response = self.model(messages)
            parts    = response.candidates[0].content.parts

            tool_calls = [
                {"name": p.function_call.name, "args": dict(p.function_call.args)}
                for p in parts
                if getattr(p, "function_call", None) is not None
            ]

            if not tool_calls:
                return response.text

            print(f"  Tools called: {[tc['name'] for tc in tool_calls]}")
            messages.append(response.candidates[0].content)

            tool_results = []
            for tc in tool_calls:
                fn     = self.tool_definitions[tc["name"]]
                result = fn(**tc["args"]) if tc["name"] in self.tool_definitions \
                         else f"Error: unknown tool '{tc['name']}'"
                tool_results.append({"name": tc["name"], "result": result})

            messages.append({"role": "user", "parts": [
                {"function_response": {"name": tr["name"], "response": {"result": tr["result"]}}}
                for tr in tool_results
            ]})

        return "Max steps reached."

Add tools and skills to the model

In [54]:
ci_skill = CompetitorIntelligenceSkill(
    tool_definitions=TOOLS,
    max_steps=12
)
print("CompetitorIntelligenceSkill ready.")

CompetitorIntelligenceSkill ready.


---
## Run the Skill


In [55]:
print("BATTLE CARD: NexaCRM vs Hartwell Financial Group deal")
print("=" * 65)
battle_card = ci_skill.run(
     vendor="NexaCRM",
     deal_context="$500K enterprise CRM, 200+ seats, financial services, Azure AD SSO required")
print("\n" + "=" * 65)
print(battle_card)

BATTLE CARD: NexaCRM vs Hartwell Financial Group deal
  [Skill step 1] Calling LLM...
  Tools called: ['get_product_pricing', 'get_product_pricing', 'get_feature_analysis', 'get_feature_analysis', 'get_feature_analysis', 'get_recent_news', 'get_customer_sentiment', 'get_customer_sentiment']
  [Skill step 2] Calling LLM...

NexaCRM Battle Card

**1. Pricing & Packaging**

*   **Enterprise Tier:** Custom pricing, estimated $120-150/user/month.
    *   Includes: All Professional features, SSO (SAML/OIDC), SLA (99.9%), dedicated Customer Success Manager (CSM), audit logs, custom data retention, multi-region support.
    *   **AI Features:** Add-on cost of $15/user/month (Deal Scoring, Email Suggestions).
*   **Professional Tier:** $89/user/month.
    *   Includes: Core features, API access, advanced pipeline analytics, key connectors (Salesforce, HubSpot), Slack/Jira.
    *   **Excludes:** SSO, SLA, dedicated CSM, enterprise-grade security controls.
*   **Contextual Notes:** Deal is for $5

---
## Skill Reusability: Run Against a Different Vendor

The same skill, zero code changes. Only the vendor argument changes.


In [ ]:
# Uncomment to run against a second vendor

# battle_card_2 = ci_skill.run(
#     vendor="SalesforceCRM",
#     deal_context="$500K enterprise CRM, 200+ seats, financial services, Azure AD SSO required"
# )
# print(battle_card_2)

print("To run against another vendor, change 'NexaCRM' to the target vendor name.")
print("The skill, tools, and loop are unchanged.")

To run against another vendor, change 'NexaCRM' to the target vendor name.
The skill, tools, and loop are unchanged.


## Summary

1. **Put constraints in the description.** The LLM reads descriptions, not the function body. List valid enum values explicitly: `"Valid values: starter, professional, enterprise."` The LLM uses this to self-correct before making the call.

2. **Return structured errors, not exceptions.** If the tool receives `tier='premium'`, return `VALIDATION_ERROR: ...` as a string. The agent can read this and retry with a valid value. An unhandled Python exception crashes the loop.

3. **Precise descriptions prevent tool confusion.** If two tools have overlapping descriptions, the LLM will choose incorrectly. One tool = one distinct purpose.

4. **Agent skills encapsulate reusable capability.** A skill packages tools + prompt + config into a single object. It can be tested in isolation, composed with other skills, and maintained by a single owner.

5. **Skills are the building blocks of multi-agent systems.** When talking about complex systems, we'll see
   orchestrator agents that delegate subtasks to specialised skills.